In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# Load dataset
df = pd.read_csv('fake_job_postings.csv')

# Fill missing and create combined text
df.fillna("", inplace=True)
df["text"] = df["title"] + " " + df["company_profile"] + " " + df["description"] + " " + df["requirements"] + " " + df["benefits"]

# Lowercase and clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # remove URLs
    text = re.sub(r"\S+@\S+", '', text)  # remove emails
    text = re.sub(r"[^a-z\s]", '', text)  # remove non-letters
    return text

df["clean_text"] = df["text"].apply(clean_text)

# Add scam keyword feature
scam_keywords = ["no experience required", "work from home", "work-at-home", "immediate start", "entry level", "quick hire", "start today", "flexible hours", "part-time opportunity", "freelance only", "earn $$$", "make money fast", "unlimited income", "high paying", "cash daily", "weekly payout", "get paid instantly", "easy money", "residual income", "financial freedom", "apply now", "limited slots", "urgent hiring", "act fast", "don’t miss out", "apply immediately", "today only", "exclusive offer", "click here", "sign up now", "visit our site", "send your CV here", "contact us at", "call this number", "WhatsApp only", "Telegram for more", "no skills needed", "instant approval", "guaranteed job", "be your own boss", "100% legit", "verified opportunity", "get rich quick", "social media agent", "online posting agent", "data entry operator", "virtual assistant needed urgently", "online evaluator", "online reviewer"]
def contains_scam_words(text):
    return int(any(word in text for word in scam_keywords))

df["scam_flag"] = df["clean_text"].apply(contains_scam_words)

# Vectorize with bigrams
vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words="english", max_features=4000)
X_tfidf = vectorizer.fit_transform(df["clean_text"])

# Combine with scam_flag
X = np.hstack((X_tfidf.toarray(), df["scam_flag"].values.reshape(-1, 1)))
y = df["fraudulent"]

# Resample with SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Split and train
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=150, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("\n📊 Evaluation Report:")
print(classification_report(y_test, y_pred))

# Command-line like input (in Jupyter)
def test_job_cli():
    print("\n💼 Scam Job Checker (type 'exit' to stop)")
    while True:
        inp = input("\nEnter a job description:\n")
        if inp.lower() == "exit":
            break
        inp_clean = clean_text(inp)
        scam_feature = int(contains_scam_words(inp_clean))
        vec = vectorizer.transform([inp_clean])
        combined = np.hstack((vec.toarray(), [[scam_feature]]))
        pred = model.predict(combined)[0]
        print("🔴 FAKE JOB\n" if pred == 1 else "🟢 REAL JOB\n")

test_job_cli()



📊 Evaluation Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3396
           1       1.00      1.00      1.00      3410

    accuracy                           1.00      6806
   macro avg       1.00      1.00      1.00      6806
weighted avg       1.00      1.00      1.00      6806


💼 Scam Job Checker (type 'exit' to stop)



Enter a job description:
     "Join our dynamic marketing team as a content strategist. Requirements include a bachelor's degree in marketing and strong communication skills.",


🟢 REAL JOB




Enter a job description:
     "Click here now to apply! Limited slots available for a freelance online posting agent. Weekly payout guaranteed.",


🟢 REAL JOB




Enter a job description:
     "Urgent hiring: data entry operators needed immediately. No skills required. Apply now to start earning fast!",


🔴 FAKE JOB




Enter a job description:
     "Work from home and earn $$$ daily! No experience required. Start today and get paid instantly!",


🔴 FAKE JOB




Enter a job description:
 exit
